# Classificação de Hipotireoidismo com Machine Learning

## Tech Challenge - Fase 1 | IA para Devs

Este projeto apresenta uma solução inicial de Machine Learning aplicada à análise de dados médicos tabulares, com foco na classificação de registros clínicos relacionados à tireoide. A proposta consiste em estimar, a partir de variáveis demográficas, clínicas e laboratoriais, se um registro apresenta indicação de **hipotireoidismo** ou se pertence à classe **negativa**.

O problema foi estruturado como uma tarefa de classificação binária supervisionada. Esse tipo de abordagem pode contribuir para etapas de triagem, priorização de análise e apoio à organização do fluxo clínico, especialmente em cenários com grande volume de exames e dados estruturados.

A solução desenvolvida tem caráter acadêmico e experimental. Os resultados obtidos devem ser interpretados como apoio analítico à decisão médica, sem substituir avaliação clínica, anamnese, exame físico, protocolos institucionais ou julgamento profissional. A decisão final sobre diagnóstico e conduta permanece sempre sob responsabilidade de profissionais da saúde.

## 1. Preparação do Ambiente

A execução foi pensada para o Google Colab, ambiente no qual o dataset será baixado diretamente do repositório GitHub para uma pasta local da sessão. A célula abaixo permanece como recurso opcional para instalação de dependências caso o ambiente não contenha todos os pacotes necessários.

In [ ]:
# Célula opcional para Google Colab.
# Execute somente se alguma biblioteca não estiver disponível no ambiente.
#
# !pip install -q pandas numpy matplotlib seaborn scikit-learn joblib shap

## 2. Importação das Bibliotecas

Foram utilizadas bibliotecas consolidadas do ecossistema Python para análise de dados, visualização, modelagem supervisionada, avaliação de desempenho e persistência do modelo. O parâmetro `RANDOM_STATE` foi definido para favorecer reprodutibilidade nas divisões amostrais e nos algoritmos estocásticos.

In [ ]:
import os
import random
import sys
import urllib.request
import warnings
from pathlib import Path
from IPython.display import display

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, auc, classification_report, confusion_matrix,
    average_precision_score, f1_score, precision_recall_curve,
    precision_score, recall_score, roc_auc_score, roc_curve
)
from sklearn.model_selection import learning_curve, train_test_split, validation_curve
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 11

RANDOM_STATE = 42
TARGET_COLUMN = "target"
POSITIVE_LABEL = "hypothyroid"
NEGATIVE_LABEL = "negative"

In [ ]:
os.environ["PYTHONHASHSEED"] = str(RANDOM_STATE)
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

print(f"Seed global definida: {RANDOM_STATE}")

## 3. Fonte de Dados e Diretórios do Projeto

O arquivo `hypothyroid_final.csv` será obtido diretamente do GitHub e armazenado na estrutura `/content/TechCahllenge_Tireoide/dataset/` quando o notebook estiver no Google Colab. Em execução local, o código mantém compatibilidade com o arquivo já presente no repositório.

In [ ]:
GITHUB_DATA_URL = (
    "https://raw.githubusercontent.com/"
    "mvaraujo1977/TechCahllenge_Tireoide/"
    "Nirton_Afonso/dataset/hypothyroid_final.csv"
)

RUNNING_IN_COLAB = "google.colab" in sys.modules

if RUNNING_IN_COLAB:
    PROJECT_ROOT = Path("/content/TechCahllenge_Tireoide")
    DATA_DIR = PROJECT_ROOT / "dataset"
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    DATA_PATH = DATA_DIR / "hypothyroid_final.csv"

    print("Ambiente Google Colab detectado.")
    print(f"Baixando dataset do GitHub para: {DATA_PATH}")
    urllib.request.urlretrieve(GITHUB_DATA_URL, DATA_PATH)
else:
    PROJECT_ROOT = Path.cwd()
    if PROJECT_ROOT.name.lower() == "notebooks":
        PROJECT_ROOT = PROJECT_ROOT.parent

    DATA_PATH = PROJECT_ROOT / "dataset" / "hypothyroid_final.csv"
    if not DATA_PATH.exists():
        DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
        print("Dataset local não encontrado. Baixando do GitHub para a pasta dataset/ do projeto.")
        urllib.request.urlretrieve(GITHUB_DATA_URL, DATA_PATH)

MODELS_DIR = PROJECT_ROOT / "models"
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f"Fonte GitHub: {GITHUB_DATA_URL}")
print(f"Dataset em uso: {DATA_PATH.resolve()}")
print(f"Diretório de modelos: {MODELS_DIR.resolve()}")
print(f"Diretório de figuras: {FIGURES_DIR.resolve()}")

## 4. Carregamento do Dataset

A base utilizada corresponde à versão tabular já consolidada do conjunto de dados de hipotireoidismo. A variável `target` representa a classe de interesse e será utilizada como variável-alvo do problema de classificação.

In [ ]:
dados = pd.read_csv(DATA_PATH, na_values=["?", "", "NA", "NaN"])

print("Primeiras linhas:")
display(dados.head())
print(f"Dimensão: {dados.shape[0]} linhas e {dados.shape[1]} colunas")
print("\nColunas:")
print(list(dados.columns))

In [ ]:
print("Informações gerais:")
dados.info()

print("\nEstatísticas numéricas:")
display(dados.describe().T)

print("\nEstatísticas categóricas:")
display(dados.describe(include="object").T)

A inspeção inicial evidencia a presença de variáveis contínuas, como idade e exames laboratoriais, e variáveis categóricas associadas a características clínicas, uso de medicamentos, sintomas e indicação de exames medidos. Essa composição reforça a adequação de pipelines de pré-processamento distintos para atributos numéricos e categóricos.

## 5. Análise Exploratória dos Dados (EDA)

A análise exploratória foi conduzida para caracterizar a qualidade, a estrutura e os padrões gerais da base. Em dados médicos, essa etapa é particularmente relevante, pois ausência de exames, desbalanceamento de classes e valores extremos podem refletir tanto limitações operacionais da coleta quanto sinais clínicos relevantes.

In [ ]:
linhas, colunas = dados.shape
print(f"Linhas: {linhas}")
print(f"Colunas: {colunas}")

resumo_tipos = dados.dtypes.value_counts().rename_axis("tipo").reset_index(name="quantidade")
display(resumo_tipos)

plt.figure(figsize=(7, 4))
sns.barplot(data=resumo_tipos, x="tipo", y="quantidade")
plt.title("Quantidade de variáveis por tipo de dado")
plt.xlabel("Tipo de dado")
plt.ylabel("Quantidade de colunas")
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()

A distribuição dos tipos de dados confirma a presença de variáveis numéricas e categóricas. Essa distinção será preservada no pré-processamento, permitindo imputação, padronização e codificação de forma apropriada para cada grupo de atributos.

In [ ]:
missing_count = dados.isna().sum().sort_values(ascending=False)
missing_percent = (missing_count / len(dados) * 100).round(2)
missing_table = pd.DataFrame({"ausentes": missing_count, "percentual": missing_percent})
missing_table = missing_table[missing_table["ausentes"] > 0]
display(missing_table)

plt.figure(figsize=(12, 5))
sns.heatmap(dados.isna(), cbar=False, cmap="viridis", yticklabels=False)
plt.title("Mapa de valores ausentes no dataset")
plt.xlabel("Variáveis")
plt.ylabel("Registros")
plt.tight_layout()
plt.show()

if not missing_table.empty:
    plt.figure(figsize=(10, 5))
    sns.barplot(data=missing_table.reset_index(), x="percentual", y="index", color="#5DADE2")
    plt.title("Percentual de valores ausentes por variável")
    plt.xlabel("Percentual de valores ausentes (%)")
    plt.ylabel("Variável")
    plt.tight_layout()
    plt.show()

Os valores ausentes concentram-se principalmente em variáveis laboratoriais. Esse padrão é compatível com bases clínicas, nas quais nem todos os exames são solicitados para todos os pacientes. Para evitar vazamento de informação, a imputação será ajustada exclusivamente com os dados de treino dentro do pipeline.

In [ ]:
duplicados = dados.duplicated().sum()
print(f"Registros duplicados encontrados: {duplicados}")
print(f"Percentual de duplicados: {duplicados / len(dados) * 100:.2f}%")

A presença de registros duplicados pode introduzir viés na avaliação do modelo, sobretudo se uma mesma observação aparecer em subconjuntos diferentes. Por esse motivo, a duplicidade será tratada explicitamente na etapa de limpeza.

In [ ]:
target_counts = dados[TARGET_COLUMN].value_counts()
target_percent = dados[TARGET_COLUMN].value_counts(normalize=True).mul(100).round(2)
target_summary = pd.DataFrame({"quantidade": target_counts, "percentual": target_percent})
display(target_summary)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.countplot(data=dados, x=TARGET_COLUMN, order=target_counts.index, ax=axes[0])
axes[0].set_title("Distribuição da variável-alvo")
axes[0].set_xlabel("Classe")
axes[0].set_ylabel("Quantidade")
axes[1].pie(target_counts.values, labels=target_counts.index, autopct="%1.1f%%", startangle=90)
axes[1].set_title("Proporção das classes")
plt.tight_layout()
plt.show()

A variável-alvo apresenta desbalanceamento relevante, com predominância da classe negativa. Esse aspecto influencia a avaliação dos modelos: métricas como recall e F1-score são mais informativas que a acurácia isolada, especialmente quando a classe positiva possui maior importância clínica.

In [ ]:
numeric_columns = dados.select_dtypes(include=[np.number]).columns.tolist()
categorical_columns = [col for col in dados.columns if col not in numeric_columns and col != TARGET_COLUMN]

print("Variáveis numéricas:", numeric_columns)
print("\nVariáveis categóricas:", categorical_columns)

In [ ]:
if numeric_columns:
    n_cols = 3
    n_rows = int(np.ceil(len(numeric_columns) / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4 * n_rows))
    axes = np.array(axes).reshape(-1)
    for ax, col in zip(axes, numeric_columns):
        sns.histplot(data=dados, x=col, kde=True, ax=ax, color="#4C78A8")
        ax.set_title(f"Distribuição de {col}")
        ax.set_xlabel(col)
        ax.set_ylabel("Frequência")
    for ax in axes[len(numeric_columns):]:
        ax.set_visible(False)
    plt.tight_layout()
    plt.show()

As distribuições numéricas indicam assimetrias e concentrações específicas em variáveis laboratoriais. Em exames clínicos, esse comportamento é esperado e pode refletir tanto padrões populacionais quanto alterações associadas à condição investigada.

In [ ]:
if numeric_columns:
    n_cols = 3
    n_rows = int(np.ceil(len(numeric_columns) / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4 * n_rows))
    axes = np.array(axes).reshape(-1)
    for ax, col in zip(axes, numeric_columns):
        sns.boxplot(data=dados, x=TARGET_COLUMN, y=col, ax=ax, palette="Set3")
        ax.set_title(f"Boxplot de {col} por classe")
        ax.set_xlabel("Classe")
        ax.set_ylabel(col)
        ax.tick_params(axis="x", rotation=20)
    for ax in axes[len(numeric_columns):]:
        ax.set_visible(False)
    plt.tight_layout()
    plt.show()

In [ ]:
selected_numeric = [col for col in ["age", "TSH", "T3", "TT4", "T4U", "FTI"] if col in dados.columns]

if selected_numeric:
    n_cols = 2
    n_rows = int(np.ceil(len(selected_numeric) / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 4.5 * n_rows))
    axes = np.array(axes).reshape(-1)
    for ax, col in zip(axes, selected_numeric):
        sns.violinplot(data=dados, x=TARGET_COLUMN, y=col, inner="quartile", ax=ax, palette="Pastel1")
        ax.set_title(f"Violin plot de {col} por classe")
        ax.set_xlabel("Classe")
        ax.set_ylabel(col)
        ax.tick_params(axis="x", rotation=20)
    for ax in axes[len(selected_numeric):]:
        ax.set_visible(False)
    plt.tight_layout()
    plt.show()

Os boxplots e violin plots permitem comparar dispersão, mediana e amplitude das variáveis por classe. Valores extremos foram avaliados com cautela, pois podem representar achados laboratoriais clinicamente relevantes e não apenas erros de registro.

In [ ]:
selected_categorical = [
    col for col in [
        "sex", "on_thyroxine", "query_hypothyroid", "query_hyperthyroid",
        "pregnant", "sick", "tumor", "thyroid_surgery", "TSH_measured", "T3_measured"
    ] if col in dados.columns
]

if selected_categorical:
    n_cols = 2
    n_rows = int(np.ceil(len(selected_categorical) / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 4.2 * n_rows))
    axes = np.array(axes).reshape(-1)
    for ax, col in zip(axes, selected_categorical):
        sns.countplot(data=dados, x=col, hue=TARGET_COLUMN, ax=ax)
        ax.set_title(f"Distribuição de {col} por classe")
        ax.set_xlabel(col)
        ax.set_ylabel("Quantidade")
        ax.tick_params(axis="x", rotation=20)
        ax.legend(title="Classe")
    for ax in axes[len(selected_categorical):]:
        ax.set_visible(False)
    plt.tight_layout()
    plt.show()

In [ ]:
if selected_categorical:
    percent_frames = []
    for col in selected_categorical:
        temp = (
            pd.crosstab(dados[col], dados[TARGET_COLUMN], normalize="index")
            .mul(100)
            .reset_index()
            .melt(id_vars=col, var_name="classe", value_name="percentual")
            .rename(columns={col: "categoria"})
        )
        temp["variavel"] = col
        percent_frames.append(temp)

    categorical_percent = pd.concat(percent_frames, ignore_index=True)
    display(categorical_percent.head(20))

    plt.figure(figsize=(13, 7))
    plot_data = categorical_percent[categorical_percent["classe"] == POSITIVE_LABEL]
    sns.barplot(data=plot_data, x="percentual", y="variavel", hue="categoria")
    plt.title("Percentual da classe positiva por categoria")
    plt.xlabel("Percentual de hipotireoidismo dentro da categoria (%)")
    plt.ylabel("Variável")
    plt.legend(title="Categoria", bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.show()

As comparações categóricas indicam como características clínicas e indicadores de exames medidos se distribuem entre as classes. As diferenças observadas devem ser entendidas como associações exploratórias, não como evidência causal.

In [ ]:
scatter_features = [col for col in ["TSH", "T3", "TT4", "T4U", "FTI", "age"] if col in dados.columns]

if len(scatter_features) >= 2:
    sns.pairplot(
        dados[[TARGET_COLUMN] + scatter_features].dropna(),
        hue=TARGET_COLUMN,
        diag_kind="kde",
        corner=True,
        plot_kws={"alpha": 0.55, "s": 22},
    )
    plt.suptitle("Relações entre variáveis laboratoriais selecionadas", y=1.02)
    plt.show()

In [ ]:
correlation_data = dados.copy()
correlation_data["target_binario"] = correlation_data[TARGET_COLUMN].map({NEGATIVE_LABEL: 0, POSITIVE_LABEL: 1})
corr_columns = numeric_columns + ["target_binario"]
corr_matrix = correlation_data[corr_columns].corr(numeric_only=True)

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True)
plt.title("Heatmap de correlação entre variáveis numéricas")
plt.tight_layout()
plt.show()

corr_target = (
    corr_matrix["target_binario"]
    .drop("target_binario")
    .dropna()
    .sort_values(key=lambda s: s.abs(), ascending=False)
)
display(corr_target.to_frame("correlacao_com_alvo"))

plt.figure(figsize=(9, 5))
sns.barplot(x=corr_target.values, y=corr_target.index, palette="vlag")
plt.axvline(0, color="black", linewidth=1)
plt.title("Ranking de correlação com a classe positiva")
plt.xlabel("Correlação com target binário")
plt.ylabel("Variável numérica")
plt.tight_layout()
plt.show()

A matriz de correlação resume associações lineares entre variáveis numéricas e a classe positiva codificada. Embora esse diagnóstico seja útil para orientar a interpretação inicial, relações clínicas podem ser não lineares e dependentes de interações entre atributos.

## 6. Limpeza dos Dados

A limpeza adotou uma postura conservadora, adequada ao contexto médico. Registros duplicados foram removidos para reduzir viés amostral, enquanto valores extremos foram mantidos por sua possível relevância clínica. O tratamento de ausentes foi reservado ao pipeline de modelagem.

In [ ]:
dados_limpos = dados.copy()
antes = len(dados_limpos)
dados_limpos = dados_limpos.drop_duplicates().reset_index(drop=True)
depois = len(dados_limpos)

print(f"Registros antes: {antes}")
print(f"Registros após remoção de duplicados: {depois}")
print(f"Duplicados removidos: {antes - depois}")

dados_limpos = dados_limpos[dados_limpos[TARGET_COLUMN].isin([NEGATIVE_LABEL, POSITIVE_LABEL])].reset_index(drop=True)

print("Distribuição após limpeza:")
display(dados_limpos[TARGET_COLUMN].value_counts().to_frame("quantidade"))

In [ ]:
checks = []
if "age" in dados_limpos.columns:
    checks.append({
        "variavel": "age",
        "menor_que_zero": int((dados_limpos["age"] < 0).sum()),
        "maior_que_120": int((dados_limpos["age"] > 120).sum()),
    })

for col in ["TSH", "T3", "TT4", "T4U", "FTI", "TBG"]:
    if col in dados_limpos.columns:
        checks.append({
            "variavel": col,
            "menor_que_zero": int((dados_limpos[col] < 0).sum()),
            "maior_que_120": np.nan,
        })

display(pd.DataFrame(checks))

Após a remoção de duplicados, a base permanece com a estrutura necessária para modelagem supervisionada. A decisão de manter outliers evita a exclusão indevida de casos potencialmente importantes do ponto de vista clínico.

## 7. Separação entre Variáveis Preditoras e Variável-Alvo

As variáveis preditoras correspondem aos atributos disponíveis para estimar a classe do registro. A variável-alvo representa o desfecho de interesse. A separação entre `X` e `y` formaliza o problema supervisionado e prepara os dados para treinamento e avaliação dos modelos.

In [ ]:
X = dados_limpos.drop(columns=[TARGET_COLUMN])
y = dados_limpos[TARGET_COLUMN].map({NEGATIVE_LABEL: 0, POSITIVE_LABEL: 1})

print(f"Formato de X: {X.shape}")
print(f"Formato de y: {y.shape}")
print("Mapeamento: 0 = negative, 1 = hypothyroid")
display(y.value_counts().rename(index={0: NEGATIVE_LABEL, 1: POSITIVE_LABEL}).to_frame("quantidade"))

## 8. Separação em Treino, Validação e Teste

A base foi dividida em 70% para treino, 15% para validação e 15% para teste, com estratificação para preservar a proporção entre classes. O conjunto de validação será utilizado para comparação e escolha do modelo; o teste ficará reservado para a estimativa final de generalização.

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=RANDOM_STATE, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=RANDOM_STATE, stratify=y_temp
)

splits_summary = pd.DataFrame({
    "conjunto": ["treino", "validação", "teste"],
    "quantidade": [len(X_train), len(X_val), len(X_test)],
    "percentual": [len(X_train)/len(X), len(X_val)/len(X), len(X_test)/len(X)],
    "positivos": [int(y_train.sum()), int(y_val.sum()), int(y_test.sum())],
})
splits_summary["percentual"] = (splits_summary["percentual"] * 100).round(2)
display(splits_summary)

## 9. Pipeline de Pré-processamento

O pré-processamento foi encapsulado em pipelines do `scikit-learn` para reduzir risco de data leakage. Assim, imputação, padronização e codificação categórica são ajustadas somente no conjunto de treino e aplicadas de forma consistente aos demais subconjuntos.

In [ ]:
numeric_features = X_train.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=[np.number]).columns.tolist()

print("Variáveis numéricas:", numeric_features)
print("Variáveis categóricas:", categorical_features)

def make_one_hot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def build_preprocessor(scale_numeric=True):
    numeric_steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale_numeric:
        numeric_steps.append(("scaler", StandardScaler()))

    numeric_transformer = Pipeline(numeric_steps)
    categorical_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", make_one_hot_encoder()),
    ])

    return ColumnTransformer([
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ])

## 10. Modelagem

Foram avaliadas três abordagens de classificação: Regressão Logística, Random Forest e HistGradientBoostingClassifier. A Regressão Logística funciona como baseline interpretável; a Random Forest captura relações não lineares; e o modelo de boosting incorpora `early_stopping`, interrompendo o treinamento quando não há ganho consistente na validação interna.

In [ ]:
modelos = {
    "Regressão Logística": Pipeline([
        ("preprocessor", build_preprocessor(scale_numeric=True)),
        ("classifier", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE)),
    ]),
    "Random Forest": Pipeline([
        ("preprocessor", build_preprocessor(scale_numeric=False)),
        ("classifier", RandomForestClassifier(
            n_estimators=400, min_samples_leaf=2, class_weight="balanced_subsample",
            random_state=RANDOM_STATE, n_jobs=-1
        )),
    ]),
    "HistGradientBoosting com Early Stopping": Pipeline([
        ("preprocessor", build_preprocessor(scale_numeric=False)),
        ("classifier", HistGradientBoostingClassifier(
            max_iter=300, learning_rate=0.05, max_leaf_nodes=31,
            early_stopping=True, validation_fraction=0.15,
            n_iter_no_change=15, random_state=RANDOM_STATE
        )),
    ]),
}

In [ ]:
def get_positive_scores(model, X_data):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X_data)[:, 1]
    if hasattr(model, "decision_function"):
        return model.decision_function(X_data)
    return None


def evaluate_model(model, X_data, y_true):
    y_pred = model.predict(X_data)
    y_score = get_positive_scores(model, X_data)
    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "auc": roc_auc_score(y_true, y_score) if y_score is not None else np.nan,
    }
    return metrics, y_pred, y_score


def plot_confusion_matrix(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(6, 5))
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=[NEGATIVE_LABEL, POSITIVE_LABEL],
        yticklabels=[NEGATIVE_LABEL, POSITIVE_LABEL],
    )
    plt.title(title)
    plt.xlabel("Classe prevista")
    plt.ylabel("Classe real")
    plt.tight_layout()
    plt.show()

In [ ]:
resultados = []
modelos_treinados = {}
validacao_predicoes = {}

for nome, pipeline in modelos.items():
    print(f"Treinando: {nome}")
    pipeline.fit(X_train, y_train)
    modelos_treinados[nome] = pipeline

    train_metrics, _, _ = evaluate_model(pipeline, X_train, y_train)
    val_metrics, y_val_pred, y_val_score = evaluate_model(pipeline, X_val, y_val)
    validacao_predicoes[nome] = {"pred": y_val_pred, "score": y_val_score}

    classifier = pipeline.named_steps["classifier"]
    resultados.append({
        "modelo": nome,
        "accuracy_treino": train_metrics["accuracy"],
        "recall_treino": train_metrics["recall"],
        "f1_treino": train_metrics["f1"],
        "accuracy_validacao": val_metrics["accuracy"],
        "precision_validacao": val_metrics["precision"],
        "recall_validacao": val_metrics["recall"],
        "f1_validacao": val_metrics["f1"],
        "auc_validacao": val_metrics["auc"],
        "iteracoes_usadas": getattr(classifier, "n_iter_", np.nan),
    })

resultados_df = pd.DataFrame(resultados).sort_values(
    by=["recall_validacao", "f1_validacao", "auc_validacao"],
    ascending=False,
).reset_index(drop=True)

display(resultados_df.style.format({col: "{:.3f}" for col in resultados_df.select_dtypes(include=[np.number]).columns}))

## 11. Avaliação dos Modelos

A avaliação considera accuracy, precision, recall, F1-score, matriz de confusão, curva ROC, AUC e curva Precision-Recall. Em contexto clínico, recall recebe atenção especial, pois falsos negativos podem deixar de sinalizar pacientes possivelmente doentes. A curva Precision-Recall foi incluída por ser especialmente útil em bases desbalanceadas, nas quais a classe positiva é minoritária.

In [ ]:
metrics_long = resultados_df.melt(
    id_vars="modelo",
    value_vars=["accuracy_validacao", "precision_validacao", "recall_validacao", "f1_validacao", "auc_validacao"],
    var_name="metrica",
    value_name="valor",
)

plt.figure(figsize=(13, 6))
sns.barplot(data=metrics_long, x="metrica", y="valor", hue="modelo")
plt.title("Comparação das métricas no conjunto de validação")
plt.xlabel("Métrica")
plt.ylabel("Valor")
plt.ylim(0, 1.05)
plt.xticks(rotation=20)
plt.legend(title="Modelo", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
for nome, valores in validacao_predicoes.items():
    print("=" * 80)
    print(nome)
    print(classification_report(y_val, valores["pred"], target_names=[NEGATIVE_LABEL, POSITIVE_LABEL], zero_division=0))
    plot_confusion_matrix(y_val, valores["pred"], f"Matriz de confusão - Validação - {nome}")

In [ ]:
plt.figure(figsize=(9, 7))
for nome, valores in validacao_predicoes.items():
    y_score = valores["score"]
    if y_score is None:
        continue
    fpr, tpr, _ = roc_curve(y_val, y_score)
    model_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, linewidth=2, label=f"{nome} (AUC = {model_auc:.3f})")

plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Aleatório")
plt.title("Curvas ROC no conjunto de validação")
plt.xlabel("Taxa de falsos positivos")
plt.ylabel("Taxa de verdadeiros positivos / Recall")
plt.legend(loc="lower right")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(9, 7))
for nome, valores in validacao_predicoes.items():
    y_score = valores["score"]
    if y_score is None:
        continue
    precision_curve, recall_curve, _ = precision_recall_curve(y_val, y_score)
    avg_precision = average_precision_score(y_val, y_score)
    plt.plot(
        recall_curve,
        precision_curve,
        linewidth=2,
        label=f"{nome} (AP = {avg_precision:.3f})",
    )

baseline = y_val.mean()
plt.axhline(baseline, linestyle="--", color="gray", label=f"Baseline positivos = {baseline:.3f}")
plt.title("Curvas Precision-Recall no conjunto de validação")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.legend(loc="lower left")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

As matrizes de confusão detalham a natureza dos erros cometidos pelos modelos. As curvas ROC mostram a separação entre classes em diferentes limiares, enquanto as curvas Precision-Recall destacam a relação entre identificação da classe positiva e confiabilidade das previsões positivas. Em uma base desbalanceada, essa leitura conjunta é mais robusta que a acurácia isolada.

## 12. Early Stopping e Análise de Overfitting

O early stopping foi aplicado ao modelo de boosting para monitorar a evolução do treinamento e interromper novas iterações quando a validação interna deixa de apresentar melhora relevante. Curvas de aprendizado e validação foram incluídas para comparar desempenho em treino e validação e identificar indícios de overfitting.

In [ ]:
hgb_model = modelos_treinados["HistGradientBoosting com Early Stopping"].named_steps["classifier"]

print(f"Iterações máximas configuradas: {hgb_model.max_iter}")
print(f"Iterações efetivamente usadas: {hgb_model.n_iter_}")
print(f"Early stopping ativado: {hgb_model.early_stopping}")

if hasattr(hgb_model, "validation_score_") and hgb_model.validation_score_ is not None:
    plt.figure(figsize=(10, 5))
    plt.plot(hgb_model.validation_score_, marker="o")
    plt.title("Evolução do score de validação interna - Early Stopping")
    plt.xlabel("Iteração")
    plt.ylabel("Score de validação interna")
    plt.tight_layout()
    plt.show()
else:
    print("O modelo não expôs validation_score_. n_iter_ ainda indica o ponto de parada.")

In [ ]:
for nome, modelo in modelos_treinados.items():
    print(f"Gerando learning curve para: {nome}")
    train_sizes, train_scores, val_scores = learning_curve(
        modelo, X_train, y_train, cv=3, scoring="f1",
        train_sizes=np.linspace(0.2, 1.0, 5), n_jobs=-1
    )

    plt.figure(figsize=(8, 5))
    plt.plot(train_sizes, train_scores.mean(axis=1), marker="o", label="Treino")
    plt.plot(train_sizes, val_scores.mean(axis=1), marker="s", label="Validação cruzada")
    plt.fill_between(train_sizes, train_scores.mean(axis=1)-train_scores.std(axis=1), train_scores.mean(axis=1)+train_scores.std(axis=1), alpha=0.15)
    plt.fill_between(train_sizes, val_scores.mean(axis=1)-val_scores.std(axis=1), val_scores.mean(axis=1)+val_scores.std(axis=1), alpha=0.15)
    plt.title(f"Learning Curve - {nome}")
    plt.xlabel("Quantidade de amostras de treino")
    plt.ylabel("F1-score")
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
rf_pipeline = modelos["Random Forest"]
param_range = [50, 100, 200, 400]
train_scores, val_scores = validation_curve(
    rf_pipeline, X_train, y_train,
    param_name="classifier__n_estimators",
    param_range=param_range, cv=3, scoring="f1", n_jobs=-1
)

plt.figure(figsize=(8, 5))
plt.plot(param_range, train_scores.mean(axis=1), marker="o", label="Treino")
plt.plot(param_range, val_scores.mean(axis=1), marker="s", label="Validação cruzada")
plt.title("Validation Curve - Random Forest")
plt.xlabel("Número de árvores")
plt.ylabel("F1-score")
plt.legend()
plt.tight_layout()
plt.show()

Diferenças expressivas entre desempenho de treino e validação indicam possível overfitting. Quando ambos os desempenhos são baixos, pode haver limitação de capacidade preditiva ou necessidade de melhor representação dos dados. O early stopping atua como mecanismo adicional de controle para modelos iterativos.

## 13. Escolha do Melhor Modelo

A seleção do modelo foi realizada exclusivamente com base no conjunto de validação. Os critérios priorizam recall, F1-score, AUC e estabilidade entre treino e validação, considerando o uso pretendido como apoio à triagem clínica.

In [ ]:
resultados_df["gap_f1_treino_validacao"] = (resultados_df["f1_treino"] - resultados_df["f1_validacao"]).abs()

ranking = resultados_df.sort_values(
    by=["recall_validacao", "f1_validacao", "auc_validacao", "gap_f1_treino_validacao"],
    ascending=[False, False, False, True],
).reset_index(drop=True)

display(ranking.style.format({col: "{:.3f}" for col in ranking.select_dtypes(include=[np.number]).columns}))

best_model_name = ranking.loc[0, "modelo"]
best_model = modelos_treinados[best_model_name]
print(f"Melhor modelo selecionado: {best_model_name}")

## 14. Avaliação Final no Conjunto de Teste

Após a escolha do modelo, o conjunto de teste foi utilizado para estimar desempenho em dados não vistos. Essa separação reduz otimismo indevido nas métricas finais e fornece uma aproximação mais honesta da capacidade de generalização.

In [ ]:
test_metrics, y_test_pred, y_test_score = evaluate_model(best_model, X_test, y_test)
test_results = pd.DataFrame([test_metrics], index=[best_model_name]).T.rename(columns={best_model_name: "valor"})
display(test_results.style.format("{:.3f}"))

print("Classification report final:")
print(classification_report(y_test, y_test_pred, target_names=[NEGATIVE_LABEL, POSITIVE_LABEL], zero_division=0))

plot_confusion_matrix(y_test, y_test_pred, f"Matriz de confusão final - {best_model_name}")

In [ ]:
if y_test_score is not None:
    fpr, tpr, _ = roc_curve(y_test, y_test_score)
    test_auc = auc(fpr, tpr)
    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, linewidth=2, label=f"{best_model_name} (AUC = {test_auc:.3f})")
    plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Aleatório")
    plt.title("Curva ROC final no conjunto de teste")
    plt.xlabel("Taxa de falsos positivos")
    plt.ylabel("Taxa de verdadeiros positivos / Recall")
    plt.legend(loc="lower right")
    plt.tight_layout()
    plt.show()
else:
    print("Modelo sem score probabilístico disponível para ROC.")

In [ ]:
if y_test_score is not None:
    precision_curve, recall_curve, _ = precision_recall_curve(y_test, y_test_score)
    avg_precision = average_precision_score(y_test, y_test_score)

    plt.figure(figsize=(8, 6))
    plt.plot(
        recall_curve,
        precision_curve,
        linewidth=2,
        label=f"{best_model_name} (AP = {avg_precision:.3f})",
    )
    baseline = y_test.mean()
    plt.axhline(baseline, linestyle="--", color="gray", label=f"Baseline positivos = {baseline:.3f}")
    plt.title("Curva Precision-Recall final no conjunto de teste")
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.legend(loc="lower left")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("Modelo sem score probabilístico disponível para Precision-Recall.")

## 15. Interpretação do Modelo

A interpretação considera importância por permutação, coeficientes quando aplicável e SHAP de forma opcional. O objetivo é identificar variáveis com maior influência nas previsões, mantendo a ressalva de que importância estatística não implica causalidade clínica.

In [ ]:
preprocessor = best_model.named_steps["preprocessor"]
classifier = best_model.named_steps["classifier"]

perm_importance = permutation_importance(
    best_model, X_val, y_val, n_repeats=10,
    random_state=RANDOM_STATE, scoring="f1", n_jobs=-1
)

importance_df = pd.DataFrame({
    "variavel": X_val.columns,
    "importancia_media": perm_importance.importances_mean,
    "importancia_desvio": perm_importance.importances_std,
}).sort_values("importancia_media", ascending=False)

display(importance_df.head(15))

plt.figure(figsize=(10, 6))
sns.barplot(data=importance_df.head(15), x="importancia_media", y="variavel", color="#7E57C2")
plt.title(f"Importância por permutação - {best_model_name}")
plt.xlabel("Queda média no F1-score ao embaralhar a variável")
plt.ylabel("Variável")
plt.tight_layout()
plt.show()

In [ ]:
def get_feature_names(preprocessor):
    feature_names = []
    if numeric_features:
        feature_names.extend(numeric_features)
    if categorical_features:
        cat_pipeline = preprocessor.named_transformers_["cat"]
        encoder = cat_pipeline.named_steps["onehot"]
        feature_names.extend(encoder.get_feature_names_out(categorical_features).tolist())
    return feature_names


if isinstance(classifier, LogisticRegression):
    feature_names = get_feature_names(preprocessor)
    coefficients = pd.DataFrame({
        "atributo": feature_names,
        "coeficiente": classifier.coef_[0],
    })
    coefficients["coeficiente_abs"] = coefficients["coeficiente"].abs()
    coefficients = coefficients.sort_values("coeficiente_abs", ascending=False)
    display(coefficients.head(20))

    plt.figure(figsize=(10, 7))
    sns.barplot(data=coefficients.head(20), x="coeficiente", y="atributo", palette="vlag")
    plt.axvline(0, color="black", linewidth=1)
    plt.title("Coeficientes mais relevantes - Regressão Logística")
    plt.xlabel("Coeficiente")
    plt.ylabel("Atributo")
    plt.tight_layout()
    plt.show()
else:
    print("O melhor modelo não é Regressão Logística; coeficientes lineares não se aplicam.")

In [ ]:
try:
    import shap

    feature_names = get_feature_names(preprocessor)
    X_val_transformed = preprocessor.transform(X_val)
    sample_size = min(250, X_val_transformed.shape[0])
    X_shap = X_val_transformed[:sample_size]

    explainer = shap.Explainer(classifier, X_shap, feature_names=feature_names)
    shap_values = explainer(X_shap)

    # Em classificadores binários, alguns explainers retornam uma dimensão extra
    # para as classes. Para este projeto, a visualização usa a classe positiva.
    if len(shap_values.values.shape) == 3:
        shap_values_plot = shap.Explanation(
            values=shap_values.values[:, :, 1],
            base_values=shap_values.base_values[:, 1],
            data=shap_values.data,
            feature_names=feature_names,
        )
    else:
        shap_values_plot = shap_values

    shap.plots.beeswarm(shap_values_plot, max_display=15)

except Exception as exc:
    print("SHAP não foi executado neste ambiente.")
    print(f"Motivo: {type(exc).__name__}: {exc}")

As variáveis mais influentes devem ser analisadas como sinais estatísticos do modelo, não como comprovação de causa. Qualquer conclusão clínica exige validação por especialistas, avaliação de protocolos e análise do contexto de coleta dos dados.

## 16. Salvamento e Carregamento do Modelo

O pipeline final será persistido com o pré-processamento e o classificador selecionado, garantindo que futuras previsões utilizem exatamente as mesmas transformações aplicadas durante o treinamento.

In [ ]:
model_path = MODELS_DIR / "modelo_classificacao_tireoide.pkl"
joblib.dump(best_model, model_path)
print(f"Modelo salvo em: {model_path.resolve()}")

In [ ]:
modelo_carregado = joblib.load(model_path)
print("Modelo carregado com sucesso.")
print(modelo_carregado)

## 17. Relatório Final e Discussão dos Resultados

Este projeto desenvolveu uma solução inicial de Machine Learning para classificação de registros clínicos relacionados à tireoide. A tarefa foi formulada como um problema de classificação binária entre **hipotireoidismo** e **classe negativa**, utilizando atributos demográficos, clínicos e laboratoriais disponíveis em formato tabular.

A análise exploratória indicou uma base com desbalanceamento relevante entre as classes, com predominância de registros negativos. Esse comportamento tornou necessária uma avaliação além da acurácia, com maior atenção ao **recall**, ao **F1-score** e à análise da matriz de confusão, especialmente por se tratar de um problema associado à triagem em saúde.

Na etapa de limpeza, registros duplicados foram removidos e valores extremos foram preservados, considerando sua possível relevância clínica. Os valores ausentes foram tratados dentro dos pipelines de modelagem, reduzindo o risco de *data leakage* entre treino, validação e teste. Essa estratégia permitiu manter uma abordagem conservadora e adequada ao contexto médico.

Foram avaliados três modelos: **Regressão Logística**, **Random Forest** e **HistGradientBoostingClassifier com early stopping**. O melhor modelo selecionado no conjunto de validação foi o **Random Forest**, considerando principalmente recall, F1-score, AUC e estabilidade entre treino e validação.

Na avaliação final com o conjunto de teste, o modelo obteve os seguintes resultados:

- **Accuracy:** 0.991
- **Precision:** 0.905
- **Recall:** 0.905
- **F1-score:** 0.905
- **AUC:** 0.996

Os resultados indicam desempenho elevado no conjunto de teste, com bom equilíbrio entre precision e recall para a classe positiva. O recall de **0.905** mostra que o modelo conseguiu identificar a maior parte dos registros associados a hipotireoidismo, ponto importante em um cenário de triagem. Ainda assim, a existência de falsos negativos deve ser considerada com cautela, pois representa casos potencialmente positivos que não seriam sinalizados pelo modelo.

A matriz de confusão final complementa essa análise ao permitir observar a distribuição entre acertos, falsos positivos e falsos negativos. Em contexto médico, falsos negativos podem representar risco de atraso na investigação clínica, enquanto falsos positivos podem gerar exames adicionais, custos e preocupação desnecessária. Por esse motivo, a escolha do modelo não deve considerar apenas métricas agregadas, mas também o impacto prático dos diferentes tipos de erro.

A interpretação por importância de variáveis indicou maior influência das seguintes variáveis: **FTI**, **TSH**, **TT4**, **age** e **on_thyroxine**. Esses atributos são coerentes com o contexto do problema, pois incluem exames laboratoriais relacionados à função tireoidiana, idade do paciente e uso de medicação associada à tireoide. No entanto, essa interpretação representa influência estatística no comportamento do modelo e não deve ser entendida como evidência causal.

O modelo pode apoiar uma etapa inicial de triagem ou priorização de análise, auxiliando na identificação de registros que merecem maior atenção. Entretanto, ele não substitui avaliação médica. A decisão final sobre diagnóstico e conduta deve permanecer sempre com profissionais da saúde, considerando histórico clínico, exame físico, protocolos institucionais e exames complementares.

Como limitações, este estudo utilizou uma base específica e não contempla validação externa, avaliação prospectiva, auditoria de vieses populacionais, calibração clínica formal ou integração com sistemas reais de saúde. Portanto, os resultados devem ser interpretados como uma prova de conceito acadêmica, e não como uma solução pronta para uso hospitalar.

Como melhorias futuras, recomenda-se avaliar ajuste de limiar de decisão, calibração de probabilidades, técnicas adicionais para tratamento de desbalanceamento, validação com outras bases, análise de vieses e monitoramento contínuo de desempenho. A conclusão principal é que o modelo apresenta potencial como apoio analítico em saúde, desde que utilizado com validação adequada, supervisão profissional e sem substituir a decisão clínica.